In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import sys

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [2]:
ANALYSIS_DIR = ROOT / 'data' / 'processed' / 'analysis'
OUTPUT_TABLES = ROOT / 'reports' / 'descriptives'
OUTPUT_FIGURES = ROOT / 'reports' / 'figures'

OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

WIDE_FILE = ANALYSIS_DIR / 'analysis_dataset_wide.csv'
LONG_FILE = ANALYSIS_DIR / 'all_signals_long.csv'

PRIMARY_SIGNALS = [
    't_neck',
    't_chest',
    't_abdomen',
    't_dorsal',
    't_hand',
    't_shin',
    't_microclimatefront',
    't_microclimateback',
    't_environment',
    't_core',
    'rh_neck',
    'rh_chest',
    'rh_abdomen',
    'rh_dorsal',
    'rh_hand',
    'rh_shin',
    'rh_microclimatefront',
    'rh_microclimateback',
    'rh_environment',
    'hr',
    't_microclimate_gradient',
    'rh_microclimate_gradient',
]

PHASE_ORDER = [
    "standing_1",
    "walk_low_1",
    "walk_mod_1",
    "walk_low_2",
    "walk_mod_2",
    "standing_2",
]

CONDITION_ORDER = ['no_wind', 'wind_low', 'wind_high']

In [3]:
from unittest import signals

def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not WIDE_FILE.exists():
        raise FileNotFoundError(f'Missing file: {WIDE_FILE}')
    if not LONG_FILE.exists():
        raise FileNotFoundError(f'Missing file: {LONG_FILE}')

    df_wide = pd.read_csv(WIDE_FILE)
    df_long = pd.read_csv(LONG_FILE)

    # conversioni numeriche importanti
    for col in ['elapsed_minutes', 'elapsed_seconds', 'time_bin_5min']:
        if col in df_wide.columns:
            df_wide[col] = pd.to_numeric(df_wide[col], errors='coerce')

    if 'value' in df_long.columns:
        df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')
    if 'elapsed_minutes' in df_long.columns:
        df_long['elapsed_minutes'] = pd.to_numeric(df_long['elapsed_minutes'], errors='coerce')

    # converti anche i segnali principali a numerico
    for col in df_wide.columns:
        if col not in ['subject_id', 'condition', 'phase', 'timestamp']:
            df_wide[col] = pd.to_numeric(df_wide[col], errors='ignore')

    if 'phase' in df_wide.columns:
        df_wide['phase'] = pd.Categorical(df_wide['phase'], categories=PHASE_ORDER, ordered=True)
    if 'condition' in df_wide.columns:
        df_wide['condition'] = df_wide['condition'].astype(str).str.strip()
        df_wide['condition'] = pd.Categorical(df_wide['condition'], categories=CONDITION_ORDER, ordered=True)
    if 'condition' in df_long.columns:
        df_long['condition'] = df_long['condition'].astype(str).str.strip()
        df_long['condition'] = pd.Categorical(df_long['condition'], categories=CONDITION_ORDER, ordered=True)

    return df_wide, df_long


def get_available_signals(df_wide: pd.DataFrame) -> list[str]:
    return [signal for signal in PRIMARY_SIGNALS if signal in df_wide.columns]


def save_overall_descriptives(df_wide: pd.DataFrame, signals: list[str]) -> None:
    rows = []
    for signal in signals:
        grouped = df_wide.groupby('condition', observed=False)[signal]
        summary = grouped.agg(['count', 'mean', 'std', 'median', 'min', 'max']).reset_index()
        summary.insert(0, 'signal', signal)
        rows.append(summary)

    if rows:
        out = pd.concat(rows, ignore_index=True)
        out.to_csv(OUTPUT_TABLES / 'overall_descriptives_by_condition.csv', index=False)


def save_phase_descriptives(df_wide: pd.DataFrame, signals: list[str]) -> None:
    rows = []
    for signal in signals:
        summary = (
            df_wide.groupby(['condition', 'phase'], observed=False)[signal]
            .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
            .reset_index()
        )
        summary.insert(0, 'signal', signal)
        rows.append(summary)

    if rows:
        out = pd.concat(rows, ignore_index=True)
        out.to_csv(OUTPUT_TABLES / 'phase_descriptives_by_condition.csv', index=False)

def save_subject_phase_means(df_wide: pd.DataFrame, signals: list[str]) -> None:
    group_cols = ['subject_id', 'condition', 'phase']
    cols = [c for c in signals if c in df_wide.columns]
    out = df_wide[group_cols + cols].groupby(group_cols, observed=False).mean(numeric_only=True).reset_index()
    out.to_csv(OUTPUT_TABLES / 'subject_phase_means.csv', index=False)


def save_timebin_means(df_wide: pd.DataFrame, signals: list[str]) -> None:
    if 'time_bin_5min' not in df_wide.columns:
        return
    group_cols = ['condition', 'time_bin_5min']
    cols = [c for c in signals if c in df_wide.columns]
    out = df_wide[group_cols + cols].groupby(group_cols, observed=False).agg(['mean', 'std']).reset_index()
    out.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in out.columns]
    out.to_csv(OUTPUT_TABLES / 'timebin_5min_descriptives.csv', index=False)

def plot_individual_signal_over_time(df_wide: pd.DataFrame, signal: str) -> None:
    if signal not in df_wide.columns:
        print(f"Skipping {signal}: not in df_wide")
        return

    plot_df = df_wide[['subject_id', 'condition', 'elapsed_minutes', signal]].copy()
    plot_df['subject_id'] = plot_df['subject_id'].astype(str).str.strip()
    plot_df['condition'] = plot_df['condition'].astype(str).str.strip()
    plot_df['elapsed_minutes'] = pd.to_numeric(plot_df['elapsed_minutes'], errors='coerce')
    plot_df[signal] = pd.to_numeric(plot_df[signal], errors='coerce')

    plot_df = plot_df.dropna(subset=['subject_id', 'condition', 'elapsed_minutes', signal])

    print(f"{signal} individual plot rows:", len(plot_df))

    if plot_df.empty:
        print(f"Skipping empty individual plot for {signal}")
        return

    plt.figure(figsize=(12, 6))

    combos = (
        plot_df[['subject_id', 'condition']]
        .drop_duplicates()
        .sort_values(['subject_id', 'condition'])
    )

    n_lines = 0
    for row in combos.itertuples(index=False):
        subject = row.subject_id
        condition = row.condition

        sub = plot_df[
            (plot_df['subject_id'] == subject) &
            (plot_df['condition'] == condition)
        ].sort_values('elapsed_minutes')

        if sub.empty:
            continue

        plt.plot(
            sub['elapsed_minutes'].to_numpy(dtype=float),
            sub[signal].to_numpy(dtype=float),
            label=f'{subject} | {condition}'
        )
        n_lines += 1

    if n_lines == 0:
        print(f"No lines plotted for {signal}")
        plt.close()
        return

    for xline in [10, 20, 30, 40, 50, 60]:
        plt.axvline(x=xline, linewidth=0.8, linestyle='--')

    plt.xlabel('Elapsed minutes')
    plt.ylabel(signal)
    plt.title(f'{signal} individual time series')
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    out_path = OUTPUT_FIGURES / f'{signal}_individual_over_time.png'
    plt.savefig(out_path, dpi=300)
    plt.close()

    print(f"Saved individual plot: {out_path}")
    
    
def plot_signal_over_time(df_wide: pd.DataFrame, signal: str) -> None:
    if signal not in df_wide.columns:
        return

    plot_df = df_wide[['condition', 'elapsed_minutes', signal]].copy()
    plot_df['elapsed_minutes'] = pd.to_numeric(plot_df['elapsed_minutes'], errors='coerce')
    plot_df[signal] = pd.to_numeric(plot_df[signal], errors='coerce')
    plot_df = plot_df.dropna(subset=['condition', 'elapsed_minutes', signal])

    print(f"{signal} plot rows:", len(plot_df))

    if plot_df.empty:
        print(f"Skipping empty plot for {signal}")
        return

    grouped = (
        plot_df.groupby(['condition', 'elapsed_minutes'], observed=True)[signal]
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )

    grouped = grouped.dropna(subset=['mean']).copy()
    grouped['sem'] = grouped['std'] / grouped['count'].pow(0.5)

    print(f"{signal} grouped rows:", len(grouped))

    plt.figure(figsize=(8, 5))
    for condition in CONDITION_ORDER:
        sub = grouped[grouped['condition'] == condition].sort_values('elapsed_minutes')
        print(f"{signal} | {condition} | rows:", len(sub))
        if sub.empty:
            continue

        x = sub['elapsed_minutes'].to_numpy(dtype=float)
        y = sub['mean'].to_numpy(dtype=float)
        sem = sub['sem'].fillna(0).to_numpy(dtype=float)

        plt.plot(x, y, label=condition)
        plt.fill_between(x, y - sem, y + sem, alpha=0.2)

    for xline in [10, 20, 30, 40, 50, 60]:
        plt.axvline(x=xline, linewidth=0.8, linestyle='--')

    plt.xlabel('Elapsed minutes')
    plt.ylabel(signal)
    plt.title(f'{signal} over time')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES / f'{signal}_over_time.png', dpi=300)
    plt.close()


def plot_phase_boxplot(df_wide: pd.DataFrame, signal: str) -> None:
    if signal not in df_wide.columns or 'phase' not in df_wide.columns:
        return

    phase_means = (
        df_wide.groupby(['subject_id', 'condition', 'phase'], observed=False)[signal]
        .mean()
        .reset_index()
    )

    positions = []
    data = []
    labels = []
    pos = 1
    gap = 1

    for phase in PHASE_ORDER:
        for condition in CONDITION_ORDER:
            vals = phase_means.loc[
                (phase_means['phase'] == phase) & (phase_means['condition'] == condition), signal
            ].dropna()
            if vals.empty:
                pos += 1
                continue
            positions.append(pos)
            data.append(vals.values)
            labels.append(f'{phase}\n{condition}')
            pos += 1
        pos += gap

    if not data:
        return

def export_primary_signal_list(signals: list[str]) -> None:
    pd.DataFrame({'signal': signals}).to_csv(OUTPUT_TABLES / 'available_signals.csv', index=False)


def main() -> None:
    df_wide, df_long = load_data()
    signals = get_available_signals(df_wide)
    
    print("df_wide shape:", df_wide.shape)
    print("df_wide columns:", df_wide.columns.tolist())
    print("signals:", signals)
    
    print("condition unique:", df_wide["condition"].unique() if "condition" in df_wide.columns else "missing")
    print("phase unique:", df_wide["phase"].unique() if "phase" in df_wide.columns else "missing")
    print("elapsed_minutes dtype:", df_wide["elapsed_minutes"].dtype if "elapsed_minutes" in df_wide.columns else "missing")

    for signal in signals[:5]:
        print(signal, "non-null:", df_wide[signal].notna().sum(), "dtype:", df_wide[signal].dtype)

    if not signals:
        raise ValueError('No expected signals found in analysis_dataset_wide.csv')

    export_primary_signal_list(signals)
    save_overall_descriptives(df_wide, signals)
    save_phase_descriptives(df_wide, signals)
    save_subject_phase_means(df_wide, signals)
    save_timebin_means(df_wide, signals)


    for signal in PRIMARY_SIGNALS:
        if signal in signals:
            plot_signal_over_time(df_wide, signal)
            plot_individual_signal_over_time(df_wide, signal)

    print('Descriptive analysis completed successfully.')
    print(f'Tables saved in: {OUTPUT_TABLES}')
    print(f'Figures saved in: {OUTPUT_FIGURES}')


if __name__ == '__main__':
    main()


/tmp/ipykernel_80838/587013921.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_wide[col] = pd.to_numeric(df_wide[col], errors='ignore')


df_wide shape: (6120, 27)
df_wide columns: ['subject_id', 'condition', 'timestamp', 'elapsed_seconds', 'elapsed_minutes', 'phase', 'time_bin_5min', 'rh_abdomen', 'rh_chest', 'rh_dorsal', 'rh_environment', 'rh_hand', 'rh_microclimateback', 'rh_microclimatefront', 'rh_neck', 'rh_shin', 't_abdomen', 't_chest', 't_dorsal', 't_environment', 't_hand', 't_microclimateback', 't_microclimatefront', 't_neck', 't_shin', 't_microclimate_gradient', 'rh_microclimate_gradient']
signals: ['t_neck', 't_chest', 't_abdomen', 't_dorsal', 't_hand', 't_shin', 't_microclimatefront', 't_microclimateback', 't_environment', 'rh_neck', 'rh_chest', 'rh_abdomen', 'rh_dorsal', 'rh_hand', 'rh_shin', 'rh_microclimatefront', 'rh_microclimateback', 'rh_environment', 't_microclimate_gradient', 'rh_microclimate_gradient']
condition unique: ['no_wind', 'wind_low', 'wind_high']
Categories (3, object): ['no_wind' < 'wind_low' < 'wind_high']
phase unique: ['standing_1', 'walk_low_1', 'walk_mod_1', 'walk_low_2', 'walk_mod_2',